In [6]:
import jax
import jax.numpy as jnp
from jax import lax


@jax.jit
# Dimensionless kernel prefactor (-1.0 / ( 4.0 * (jnp.pi**3/2) * jnp.sqrt(1j) ) )
def solve_eta(a_s, dt, kernel_prefactor=(-1.0/(4.0*(jnp.pi**3/2)*jnp.sqrt(1j))) ):
    a_s = jnp.asarray(a_s, dtype=jnp.complex64)

    num_points = a_s.shape[0]
    num_steps = num_points - 1

    eta_0 = -4.0 * jnp.pi * a_s[0]
    eta = jnp.zeros_like(a_s, dtype=jnp.complex64).at[0].set(eta_0)

    l1_prefactor = 2.0 * kernel_prefactor / jnp.sqrt(dt)
    j = jnp.arange(num_steps)

    def time_step(history, k):
        diffs = history[1:] - history[:-1]

        # Only j = 0, ..., k - 2 is known. The j = k - 1 term contains eta[k]
        # and is moved into the denominator below.
        valid = j < (k - 1)
        m = k - j
        safe_m = jnp.maximum(m, 1)
        weights = jnp.where(
            valid,
            jnp.sqrt(safe_m) - jnp.sqrt(safe_m - 1),
            0.0,
        )

        known_history = jnp.sum(weights * diffs)
        numerator = -1.0 + l1_prefactor * (history[k - 1] - known_history)
        denominator = 1.0 / (4.0 * jnp.pi * a_s[k]) + l1_prefactor
        eta_k = numerator / denominator

        history = history.at[k].set(eta_k)
        return history, eta_k

    indices = jnp.arange(1, num_points)
    eta, _ = lax.scan(time_step, eta, indices)

    return eta


Array([-12.566371+0.j], dtype=complex64)

In [ ]:

# %run ./test.ipynb
# import math
# import numpy as np
# import matplotlib.pyplot as plt

# test_variable_names = ["Scattering Length"]
# array_length = 100
# random_seed = 0
# curve_types_to_test = None 
# complex_values = True 
# x_axis_name = "Time"
# plot_columns = 3
# fixed_args_before_test_arrays = () 
# fixed_args_after_test_arrays = () 

# function_under_test = solve_eta

# dataset = make_test_dataset(
#     test_variable_names,
#     length=array_length,
#     seed=random_seed,
#     curve_types=curve_types_to_test,
#     complex_values=complex_values,
# )
# test_combinations = list(zip(*(dataset[name] for name in dataset.names)))

# results = []
# for curves in test_combinations:
#     test_arrays = [curve.values for curve in curves]
#     output_array = function_under_test(
#         *fixed_args_before_test_arrays,
#         *test_arrays,
#         *fixed_args_after_test_arrays,
#     )
#     results.append((curves, output_array))

# if not results:
#     raise ValueError("No test combinations were generated.")

# ncols = min(max(1, plot_columns), len(results))
# nrows = math.ceil(len(results) / ncols)
# fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows), squeeze=False)
# axes = axes.ravel()

# for ax, (curves, y_values) in zip(axes, results):
#     y_values = np.asarray(y_values)
#     time = range(len(y_values))
#     title = ", ".join(f"{curve.name}: {curve.curve_type}" for curve in curves)
#     if np.iscomplexobj(y_values):
#         y_values = np.abs(y_values)
#     ax.plot(time, y_values, marker="o", markersize=2, linewidth=1.5)
#     ax.set_title(title)
#     ax.set_xlabel(x_axis_name)

# for ax in axes[len(results):]:
#     ax.set_visible(False)

# fig.tight_layout()